##**Projeto:** Merca Data Platform

##**Squad:** 2 | Camada Gold
### Objetivo
Promover os dados validados de categorias da Silver para a camada Gold, enriquecendo com classificação de categorias raiz e disponibilizando para consumo no SQL Server.
### Origem e Destinos
| Item | Valor |
| **Origem** | `squad2/silver/ecommerce_categorias` (Delta Lake) |
| **Destino Lake** | `squad2/gold/ecommerce_categorias` (Delta Lake — append) |
| **Destino SQL** | `squad2.gold_ecommerce_categorias` (SQL Server — append) |
| **Controle** | `gold/control/ecommerce_categorias.json` |
### Regras de Negócio Aplicadas
| # | Regra | Descrição | Saída |
| 1 | Classificação de categoria raiz | Categoria sem pai (`id_categoria_pai` nulo ou vazio) recebe `is_categoria_raiz = 'SIM'` | Campo calculado |
| 2 | Alerta de mudança estrutural | Monitora se o número de categorias raiz mudou entre lotes | Log de alerta |
### Campos Calculados na Gold
| Campo | Lógica | Valores |
| `is_categoria_raiz` | Categoria sem campo pai preenchido | `'SIM'` / `'NAO'` / `'N/A'` (se coluna pai não existir) |
| `gold_processed_at` | Timestamp de processamento na Gold | datetime |
### Dependências
| Notebook | Motivo |
| `feat_squad2_99_helpers` | `get_storage_options`, `get_squad2_client`, `SQL_OPTIONS` |
| `feat_squad2_silver_categorias` | Dados validados de origem |

In [0]:
%run ../utils/feat_squad2_99_helpers

In [0]:
from deltalake import DeltaTable, write_deltalake
import pandas as pd
from datetime import datetime
import json

TABELA = "ecommerce_categorias"
TABELA_SQL = f"gold_{TABELA}"  

path_silver  = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/silver/{TABELA}"
path_gold    = f"abfss://{SQUAD2_CONTAINER}@{ADLS_STORAGE_ACCOUNT}.dfs.core.windows.net/gold/{TABELA}"
path_control = f"control/gold/{TABELA}/control_file.json"

- Processamento, Alertas e Gravação
**Regra N1 — Classificação de Categorias Raiz:**
Detecta dinamicamente o nome da coluna pai (busca por `'pai'` ou `'parent'` no nome). Uma categoria é raiz quando `id_categoria_pai` é nulo ou vazio.
**Regra N2 — Alerta de Mudança Estrutural:**
Compara a contagem de categorias raiz do lote atual com a contagem já existente na Gold.
Qualquer variação é alertada, pois mudanças no número de categorias raiz impactam toda a árvore de navegação do e-commerce.


In [0]:
try:
    if not DeltaTable.is_deltatable(path_silver, storage_options=get_storage_options()):
        print(f" [Aviso Gold] A tabela Silver de {TABELA} ainda não foi inicializada.")
    else:
        dt_silver = DeltaTable(path_silver, storage_options=get_storage_options())
        df_pandas = dt_silver.to_pandas()
 
        squad2_client = get_squad2_client()
        file_client   = squad2_client.get_file_client(path_control)
 
        processados = set()
        if file_client.exists():
            conteudo    = file_client.download_file().readall().decode('utf-8')
            processados = set(json.loads(conteudo))
 
        df_novos_dados = df_pandas[~df_pandas['bronze_source_file'].isin(processados)].copy()
 
        if df_novos_dados.empty:
            print(" Camada Gold de Categorias em dia! Nenhum dado novo para processar.")
        else:
            print(f" Processando {len(df_novos_dados)} linhas para a Tabela Final...")
 
            # Regra N1: Identificação de Categorias Raiz
            # Detecta dinamicamente o nome da coluna pai para suportar variações de schema
            col_pai = [c for c in df_pandas.columns if 'pai' in c.lower() or 'parent' in c.lower()]
            if col_pai:
                col_pai = col_pai[0]
                df_novos_dados['is_categoria_raiz'] = df_novos_dados.apply(
                    lambda row: 'SIM' if (pd.isna(row[col_pai]) or row[col_pai] == "") else 'NAO', axis=1
                )
            else:
                df_novos_dados['is_categoria_raiz'] = "N/A"
 
            # Regra N2: Alerta de mudança no número de categorias raiz entre lotes
            # Conta raízes no lote atual e compara com o estado anterior na Gold
            raizes_lote_atual = (df_novos_dados['is_categoria_raiz'] == 'SIM').sum()
 
            if DeltaTable.is_deltatable(path_gold, storage_options=get_storage_options()):
                dt_gold_existente = DeltaTable(path_gold, storage_options=get_storage_options())
                df_gold_existente = dt_gold_existente.to_pandas()
 
                if 'is_categoria_raiz' in df_gold_existente.columns:
                    raizes_anteriores = (df_gold_existente['is_categoria_raiz'] == 'SIM').sum()
 
                    print(f"\n[N2] Categorias raiz anteriores: {raizes_anteriores}")
                    print(f"[N2] Categorias raiz neste lote: {raizes_lote_atual}")
 
                    if raizes_lote_atual != raizes_anteriores:
                        variacao = raizes_lote_atual - raizes_anteriores
                        direcao  = "adicionada(s)" if variacao > 0 else "removida(s)"
                        print(f"⚠️  [ALERTA N2] {abs(variacao)} categoria(s) raiz {direcao} em relação ao lote anterior.")
                        print(f"    Mudança estrutural detectada — verificar impacto na navegação do e-commerce.")
                    else:
                        print(f"[N2] Estrutura de categorias raiz estável.")
            else:
                # Primeira execução — apenas registra a baseline
                print(f"\n[N2] Baseline estabelecida: {raizes_lote_atual} categoria(s) raiz no primeiro lote.")
 
            df_novos_dados['gold_processed_at'] = datetime.now()
 
            # Remove timezone de colunas datetime (requisito do formato Delta)
            for col in df_novos_dados.columns:
                if pd.api.types.is_datetime64_any_dtype(df_novos_dados[col]):
                    df_novos_dados[col] = df_novos_dados[col].dt.tz_localize(None)
 
            # SINK 1: Delta Lake
            write_deltalake(path_gold, df_novos_dados, mode="append", storage_options=get_storage_options())
 
            # SINK 2: SQL Server
            try:
                df_schema_sql = spark.read \
                    .format("sqlserver") \
                    .options(**SQL_OPTIONS) \
                    .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                    .load() \
                    .limit(0)
 
                spark_df_final = spark.createDataFrame(df_novos_dados)
                for col_db in df_schema_sql.columns:
                    col_match = [c for c in spark_df_final.columns if c.lower() == col_db.lower()]
                    if col_match:
                        spark_df_final = spark_df_final.withColumnRenamed(col_match[0], col_db)
                spark_df_aligned = spark_df_final.select(*df_schema_sql.columns)
                print(f"  Tabela existente localizada. Alinhando colunas e fazendo Append...")
            except Exception:
                print(f"  Criando nova tabela: [squad2].[{TABELA_SQL}]...")
                spark_df_aligned = spark.createDataFrame(df_novos_dados)
 
            spark_df_aligned.write \
                .format("sqlserver") \
                .options(**SQL_OPTIONS) \
                .option("dbtable", f"[squad2].[{TABELA_SQL}]") \
                .mode("append") \
                .save()
 
            arquivos_atuais   = set(df_novos_dados['bronze_source_file'].unique())
            todos_processados = list(processados.union(arquivos_atuais))
            file_client.upload_data(json.dumps(todos_processados), overwrite=True)
 
            print(f"\n SUCESSO! Dados de categorias gravados em squad2.{TABELA_SQL}!")
 
except Exception as e:
    print(f" Erro no processamento: {str(e)}")
    raise